# Iceberg + MinIO Integration Test

This notebook verifies that Apache Spark is correctly configured to use Apache Iceberg and MinIO (S3) for storage.

In [ ]:
# Import the SparkSession module from pyspark.sql
from pyspark.sql import SparkSession

# Initialize a new SparkSession using the builder pattern
# The configurations we defined in spark-defaults.conf are automatically loaded here
spark = SparkSession.builder \
    .appName("Iceberg_MinIO_Test") \
    .getOrCreate()

# Set the log level to WARN to avoid cluttering the output with INFO logs
spark.sparkContext.setLogLevel("WARN")

# Print a success message to indicate the session is ready
print("Spark Session initiated successfully!")

### 1. Create Sample Data
We will create a simple dataframe representing some players to save into our Iceberg warehouse.

In [ ]:
# Define a list of tuples containing sample data for football players
data = [
    (1, "Lionel Messi", "Inter Miami", 36),
    (2, "Cristiano Ronaldo", "Al Nassr", 39),
    (3, "Kylian Mbappe", "Real Madrid", 25)
]

# Define the column names for our dataframe
columns = ["id", "name", "team", "age"]

# Create the PySpark DataFrame using the sample data and column names
df = spark.createDataFrame(data, columns)

# Display the contents of the DataFrame in the notebook
df.show()

### 2. Write Data to Iceberg
We will write this data to the `lake` catalog (which we configured to point to `datalake-warehouse` in MinIO) using the `iceberg` format.

In [ ]:
# Create a database namespace named 'football' in the 'lake' catalog if it does not exist
spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.football")

# Define the full path for the target table, including catalog, namespace, and table name
table_name = "lake.football.players"

# Write the DataFrame to storage
df.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(table_name)

# Print a confirmation message once the write operation is complete
print(f"Successfully wrote data to {table_name}!")

### 3. Read Data from Iceberg
Now let's read the data back to confirm it was saved properly in MinIO.

In [ ]:
# Read the table back from Iceberg using the previously defined table name
read_df = spark.table(table_name)

# Display the retrieved data to ensure it matches what was written
read_df.show()

### 4. Basic Iceberg Features: Schema Evolution and Updates
Iceberg allows us to update data and add columns seamlessly.

In [ ]:
# Execute an SQL command to alter the table schema by adding a 'goals' column of type INT
spark.sql(f"ALTER TABLE {table_name} ADD COLUMN goals INT")

# Execute an SQL command to update the 'goals' value for the player with id = 1
spark.sql(f"UPDATE {table_name} SET goals = 800 WHERE id = 1")

# Execute an SQL command to update the 'goals' value for the player with id = 2
spark.sql(f"UPDATE {table_name} SET goals = 850 WHERE id = 2")

# Retrieve and display the modified table to verify the updates and schema changes
spark.table(table_name).show()